# Chen et al. (2024) — code validation

Goal: run **our** N-body / mass-loss pipeline (`kdk_leapfrog` -> `get_bound_particles`)
with the **paper's** setup and check we reproduce **Fig. 11** (arXiv:2408.01496).

Test case: $(M_0, R_\mathrm{apo}, e) = (10^5\,M_\odot,\ 40\ \mathrm{kpc},\ 0.6)$, in-plane,
MWPotential2014, King $W=8$ truncated at $r_\mathrm{tid}$, $\tau=2^{-13}$, $\epsilon=1$ pc,
$m=10\,M_\odot$, 3 Gyr.

Run top to bottom. **Step 1** is the quick (~5 min) accuracy check. **Step 2** (the full
21-run sweep, ~2.3 h cold, then cached) is optional. Launch Jupyter from this folder
(`ngc6569/`) so the relative paths work.

In [ ]:
import os, hashlib, json, time
import numpy as np
import astropy.units as u
import agama
from scipy.optimize import brentq

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

from leap_frog import kdk_leapfrog
from bound_funcs import get_bound_particles

agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
time_unit = u.kpc.to(u.km) * u.s.to(u.Gyr)   # 'kpc km^-1 s' -> Gyr
print("agama G =", agama.G)

In [ ]:
# minimal helpers (clean versions of the ones in the big notebook)
def shift_to_gc(xv, center_pos, center_vel):
    return xv[:, :3] + center_pos, xv[:, 3:] + center_vel

def trajectories(bound_data, sim_data):
    mass = np.array([bd["total mass"] for bd in bound_data])
    t    = np.array([sd["time"]       for sd in sim_data])
    return {"time": t, "mass": mass}

In [ ]:
# --- paper parameters ---
FID = dict(W=8.0, kmax=13, eps_pc=1.0, m_star=10.0)
M0         = 1e5            # cluster mass [Msun]
R_APO      = 40.0          # apocenter [kpc]
ECC        = 0.6           # eccentricity
R_PERI     = R_APO * (1 - ECC) / (1 + ECC)   # = 10 kpc
T_FINAL    = -3000 * u.Myr  # 3 Gyr
DOWNSAMPLE = 20

pot_study = agama.Potential("milkyway/MWPotential2014.ini")

In [ ]:
# --- orbit IC (apocenter, in-plane e=0.6), tidal radius, King truncation ratio ---
def _Mg(R):  # spherical enclosed mass from radial force: M(<R) = R^2 |a_R| / G
    a_R = pot_study.force([R, 0.0, 0.0])[0]
    return R * R * (-a_R) / agama.G

def _orbit_min_r(v_t, tmax_gyr=3.0, n=4000):
    _, traj = agama.orbit(potential=pot_study, ic=[R_APO, 0, 0, 0, v_t, 0],
                          time=tmax_gyr / time_unit, trajsize=n)
    return np.sqrt((traj[:, :3]**2).sum(axis=1)).min()

v_circ = np.sqrt(_Mg(R_APO) * agama.G / R_APO)
v_t = brentq(lambda v: _orbit_min_r(v) - R_PERI, 0.1 * v_circ, v_circ)
center_pos = np.array([R_APO, 0.0, 0.0])
center_vel = np.array([0.0, v_t, 0.0])
_rmin = _orbit_min_r(v_t)
print(f"v_circ={v_circ:.1f}, v_t={v_t:.1f} km/s | "
      f"min r={_rmin:.2f} kpc, e={(R_APO - _rmin)/(R_APO + _rmin):.3f}")

def _tidal_radius(R, M):
    dR = 1e-3 * R
    dln = (np.log(_Mg(R + dR)) - np.log(_Mg(R - dR))) / (np.log(R + dR) - np.log(R - dR))
    return R * (M / (_Mg(R) * (3.0 - dln)))**(1.0 / 3.0)

r_tid = _tidal_radius(R_APO, M0)
print(f"r_tid = {r_tid*1000:.1f} pc")

_rgrid = np.logspace(-4, 3, 4000)
def king_trunc_ratio(W):
    p = agama.Potential(type='king', W0=W, scaleRadius=1.0, mass=1.0)
    pts = np.column_stack([_rgrid, np.zeros_like(_rgrid), np.zeros_like(_rgrid)])
    rho = p.density(pts)
    return _rgrid[rho > rho.max() * 1e-10].max()
print(f"c(W=8) = r_t/r0 = {king_trunc_ratio(8.0):.3f}")

In [ ]:
# --- the run, with on-disk caching ---
CACHE_DIR = "output/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_key(W, kmax, eps_pc, m_star, Nbody):
    payload = dict(W=float(W), kmax=int(kmax), eps_pc=float(eps_pc),
                   m_star=float(m_star), Nbody=int(Nbody), M0=float(M0),
                   R_APO=float(R_APO), ECC=float(ECC),
                   tfin_myr=float(T_FINAL.to(u.Myr).value),
                   downsample=int(DOWNSAMPLE), r_tid=round(float(r_tid), 9))
    h = hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]
    return f"{CACHE_DIR}/ml_W{W:g}_k{kmax}_eps{eps_pc:g}_m{m_star:g}_N{Nbody}_{h}.npz"

def run_mass_loss(W, kmax, eps_pc, m_star, use_cache=True):
    Nbody = int(round(M0 / m_star))            # fix total mass, vary N
    key = _cache_key(W, kmax, eps_pc, m_star, Nbody)
    if use_cache and os.path.exists(key):
        d = np.load(key)
        return d["t"], d["M"]

    scaleRadius = r_tid / king_trunc_ratio(W)  # outer radius fixed to r_tid
    pot_sat = agama.Potential(type='king', W0=W, scaleRadius=scaleRadius, mass=M0)
    df_sat  = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
    xv, mass = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)
    pos, vel = shift_to_gc(xv, center_pos, center_vel)

    tmax = -T_FINAL.to(u.Gyr).value / time_unit
    tau  = 2 ** (-kmax) * time_unit
    nt   = int(tmax / tau) + 1
    eps  = eps_pc / 1000.0

    sim_data   = kdk_leapfrog(pot_study, pos, vel, mass, nt, tau, agama.G, eps,
                              time_unit, DOWNSAMPLE, last_snapshot=False)
    bound_data = get_bound_particles(sim_data, mass)
    traj       = trajectories(bound_data, sim_data)
    t_gyr, M_over_M0 = traj["time"], traj["mass"] / traj["mass"][0]
    if use_cache:
        np.savez(key, t=t_gyr, M=M_over_M0)
    return t_gyr, M_over_M0

## Step 1 — fiducial accuracy check (~5 min)

One run at the paper's fiducial (W=8, $\tau=2^{-13}$, $\epsilon=1$ pc, $m=10$, N=10,000).
**Pass if M/M0 at 3 Gyr lands around 0.70-0.72** (the black curve in every Fig. 11 panel).

In [ ]:
t0 = time.time()
t, M = run_mass_loss(**FID)
print(f"run time: {(time.time()-t0)/60:.1f} min")
print(f"M/M0 at 3 Gyr = {M[-1]:.3f}   (paper fiducial ~0.70-0.72)")

plt.figure(figsize=(6, 4))
plt.plot(t, M, color="black", lw=2)
plt.axhspan(0.70, 0.72, color="green", alpha=0.15, label="paper endpoint")
plt.xlim(0, 3); plt.ylim(0.6, 1.0)
plt.xlabel("t (Gyr)"); plt.ylabel(r"$M(t)/M_0$")
plt.title("Fiducial check vs Chen Fig. 11")
plt.legend(); plt.tight_layout(); plt.show()

## Step 2 (optional) — full 4-panel parameter study

~2.3 h cold; every finished curve is cached, so re-runs replay in seconds.
$\epsilon=0$ (PeTar) is omitted; the $\tau$ panel is expected *not* to fully converge.

In [ ]:
results = {"W": {}, "tau": {}, "eps": {}, "m": {}}
for W in [2, 4, 6, 8, 10, 12, 14]:
    print("W", W); results["W"][W] = run_mass_loss(W, FID["kmax"], FID["eps_pc"], FID["m_star"])
for k in [10, 11, 12, 13, 14, 15]:
    print("kmax", k); results["tau"][k] = run_mass_loss(FID["W"], k, FID["eps_pc"], FID["m_star"])
for e in [2**-2, 2**-1, 2**0, 2**1, 2**2]:
    print("eps", e); results["eps"][e] = run_mass_loss(FID["W"], FID["kmax"], e, FID["m_star"])
for ms in [2**-3*10, 2**0*10, 2**3*10]:
    print("m", ms); results["m"][ms] = run_mass_loss(FID["W"], FID["kmax"], FID["eps_pc"], ms)
print("done")

In [ ]:
def _colors(values, fiducial):
    vals = sorted(values)
    norm = Normalize(vmin=min(vals), vmax=max(vals))
    return {v: ("black" if np.isclose(v, fiducial) else cm.coolwarm(norm(v))) for v in vals}

fig, axes = plt.subplots(4, 1, figsize=(6, 14), sharex=True)
fig.suptitle("Chen et al. 2024 (Fig. 11)", style="italic", size=16)

ax = axes[0]; cols = _colors(results["W"], FID["W"])
for W in sorted(results["W"]):
    t, M = results["W"][W]; ax.plot(t, M, color=cols[W], lw=1.5, label=fr"$W = {int(W)}$")

ax = axes[1]; cols = _colors(results["tau"], FID["kmax"])
for k in sorted(results["tau"]):
    t, M = results["tau"][k]
    ax.plot(t, M, color=cols[k], lw=1.5, label=fr"$\tau = 2^{{-{k}}}$ kpc km$^{{-1}}$ s")

ax = axes[2]; cols = _colors(results["eps"], FID["eps_pc"])
for e in sorted(results["eps"]):
    t, M = results["eps"][e]
    ax.plot(t, M, color=cols[e], lw=1.5, label=fr"$\epsilon = 2^{{{int(round(np.log2(e)))}}}$ pc")

ax = axes[3]; cols = _colors(results["m"], FID["m_star"])
for ms in sorted(results["m"]):
    t, M = results["m"][ms]
    ax.plot(t, M, color=cols[ms], lw=1.5,
            label=fr"$m = 2^{{{int(round(np.log2(ms/10)))}}} \times 10\ M_\odot$")

for ax in axes:
    ax.set_xlim(0, 3); ax.set_ylabel(r"$M(t)/M_0$", size=14)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=10)
axes[-1].set_xlabel(r"$t$ (Gyr)", size=14)
plt.tight_layout()
fig.savefig("output/chen_fig11_massloss.pdf", bbox_inches="tight")
plt.show()